# Dataset Audit — Amharic ACOS
Structural validity, leakage, duplicates, and annotation-consistency checks over `data/prepared/` and `data/raw_explicit/`.

In [ ]:
import json, collections, itertools, re
from pathlib import Path
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
DATA = ROOT / 'data' / 'prepared'
splits = {}
for name in ['train', 'dev', 'test']:
    with open(DATA / f'{name}.jsonl', encoding='utf-8') as f:
        splits[name] = [json.loads(l) for l in f]
for k, v in splits.items():
    nq = sum(len(r['quads']) for r in v)
    print(f'{k}: {len(v)} sentences, {nq} quads ({nq/len(v):.2f}/sent)')

In [ ]:
# 1. Structural validity: span bounds, implicit-index consistency, token/text alignment
def span_text(r, s, e):
    return ' '.join(r['tokens'][s:e])
issues = collections.Counter()
examples = collections.defaultdict(list)
for split, rows in splits.items():
    for i, r in enumerate(rows):
        toks = r['tokens']
        if ' '.join(toks) != re.sub(r'\s+', ' ', r['text'].strip()):
            issues[f'{split}:tokens!=text'] += 1
            if len(examples[f'{split}:tokens!=text']) < 3: examples[f'{split}:tokens!=text'].append(i)
        if any(t == '' for t in toks):
            issues[f'{split}:empty_token'] += 1
        if len(r['quads']) == 0:
            issues[f'{split}:no_quads'] += 1
        seen_q = set()
        for q in r['quads']:
            for pre in ('a', 'o'):
                s_, e_ = q[f'{pre}_start'], q[f'{pre}_end']
                if (s_ == -1) != (e_ == -1):
                    issues[f'{split}:{pre}_half_implicit'] += 1
                if s_ != -1 and not (0 <= s_ < e_ <= len(toks)):
                    issues[f'{split}:{pre}_span_out_of_bounds'] += 1
                    if len(examples[f'{split}:{pre}_span_out_of_bounds']) < 3: examples[f'{split}:{pre}_span_out_of_bounds'].append((i, s_, e_, len(toks)))
            key = (q['a_start'], q['a_end'], q['category'], q['sentiment'], q['o_start'], q['o_end'])
            if key in seen_q:
                issues[f'{split}:duplicate_quad_in_sentence'] += 1
                if len(examples[f'{split}:duplicate_quad_in_sentence']) < 3: examples[f'{split}:duplicate_quad_in_sentence'].append(i)
            seen_q.add(key)
print('STRUCTURAL ISSUES:')
for k, v in sorted(issues.items()): print(f'  {k}: {v}')
if not issues: print('  none')
print('\nexamples:', dict(examples))

In [ ]:
# 2. Duplicates within splits and leakage across splits (exact + normalized text)
def norm(t):
    return re.sub(r'\s+', ' ', t.strip())
texts = {k: [norm(r['text']) for r in v] for k, v in splits.items()}
for k, v in texts.items():
    c = collections.Counter(v)
    dups = {t: n for t, n in c.items() if n > 1}
    print(f'{k}: {len(dups)} duplicated texts, {sum(dups.values()) - len(dups)} extra rows')
print()
for a, b in itertools.combinations(splits, 2):
    overlap = set(texts[a]) & set(texts[b])
    print(f'LEAKAGE {a} ∩ {b}: {len(overlap)} shared texts')
# conflicting annotations among duplicate texts (all splits pooled)
by_text = collections.defaultdict(list)
for k, v in splits.items():
    for r in v:
        qset = frozenset((q['a_start'], q['a_end'], q['category'], q['sentiment'], q['o_start'], q['o_end']) for q in r['quads'])
        by_text[norm(r['text'])].append((k, qset))
conflict = 0; cross_split_conflict = 0; example = None
for t, anns in by_text.items():
    if len(anns) > 1:
        qsets = {a[1] for a in anns}
        if len(qsets) > 1:
            conflict += 1
            if len({a[0] for a in anns}) > 1: cross_split_conflict += 1
            if example is None: example = (t[:80], [(a[0], sorted(a[1])) for a in anns[:2]])
print(f'\nTexts appearing >1 time with CONFLICTING annotations: {conflict} (crossing splits: {cross_split_conflict})')
print('example:', example)

In [ ]:
# 3. Distributions: implicit ratios, categories, sentiments, outliers
for k, v in splits.items():
    tot = ia = io = both = 0
    cats = collections.Counter(); sents = collections.Counter()
    for r in v:
        for q in r['quads']:
            tot += 1
            aimp = q['a_start'] == -1; oimp = q['o_start'] == -1
            ia += aimp; io += oimp; both += aimp and oimp
            cats[q['category']] += 1; sents[q['sentiment']] += 1
    print(f'{k}: quads={tot} implicit-aspect={ia/tot:.1%} implicit-opinion={io/tot:.1%} both={both/tot:.1%} any={ (ia+io-both)/tot:.1%}')
    print(f'   sentiments: {dict(sents)}')
print()
cats_all = collections.Counter(q['category'] for v in splits.values() for r in v for q in r['quads'])
print(f'{len(cats_all)} categories; 10 rarest:')
for c, n in sorted(cats_all.items(), key=lambda x: x[1])[:10]: print(f'  {c}: {n}')
print('\ntop 5:')
for c, n in cats_all.most_common(5): print(f'  {c}: {n}')
lens = sorted(len(r['tokens']) for v in splits.values() for r in v)
print(f'\ntoken lengths: min={lens[0]} median={lens[len(lens)//2]} p99={lens[int(len(lens)*0.99)]} max={lens[-1]}')

In [ ]:
# 4. Annotation consistency: same surface span → how many different labels?
asp_cat = collections.defaultdict(collections.Counter)   # aspect text -> categories
op_sent = collections.defaultdict(collections.Counter)   # opinion text -> sentiments
pair_sent = collections.defaultdict(collections.Counter) # (aspect, opinion) -> sentiments
for v in splits.values():
    for r in v:
        for q in r['quads']:
            a = ' '.join(r['tokens'][q['a_start']:q['a_end']]) if q['a_start'] != -1 else None
            o = ' '.join(r['tokens'][q['o_start']:q['o_end']]) if q['o_start'] != -1 else None
            if a: asp_cat[a][q['category']] += 1
            if o: op_sent[o][q['sentiment']] += 1
            if a and o: pair_sent[(a, o)][q['sentiment']] += 1
def ambig(d, min_total=5):
    rows = [(k, c) for k, c in d.items() if sum(c.values()) >= min_total and len(c) > 1]
    rows.sort(key=lambda x: -sum(x[1].values()))
    return rows
ac = ambig(asp_cat)
print(f'aspect spans (≥5 occurrences) mapped to >1 category: {len(ac)} / {sum(1 for c in asp_cat.values() if sum(c.values())>=5)}')
for k, c in ac[:8]: print(f'  {k!r}: {dict(c.most_common(4))}')
os_ = ambig(op_sent)
print(f'\nopinion spans (≥5) with >1 sentiment: {len(os_)} / {sum(1 for c in op_sent.values() if sum(c.values())>=5)}')
for k, c in os_[:8]: print(f'  {k!r}: {dict(c.most_common(4))}')
ps = [( (a,o), c) for (a,o), c in pair_sent.items() if len(c) > 1 and sum(c.values()) >= 3]
print(f'\nSAME (aspect, opinion) pair with CONTRADICTORY sentiment (≥3 occ): {len(ps)}')
for (a, o), c in sorted(ps, key=lambda x: -sum(x[1].values()))[:6]: print(f'  ({a!r}, {o!r}): {dict(c)}')

In [ ]:
# 5. Span sanity: lengths, overlaps between aspect and opinion, category-sentiment degeneracy
alen = collections.Counter(); olen = collections.Counter(); ov = 0; tot_eo = 0
for v in splits.values():
    for r in v:
        for q in r['quads']:
            if q['a_start'] != -1: alen[q['a_end'] - q['a_start']] += 1
            if q['o_start'] != -1: olen[q['o_end'] - q['o_start']] += 1
            if q['a_start'] != -1 and q['o_start'] != -1:
                tot_eo += 1
                if max(q['a_start'], q['o_start']) < min(q['a_end'], q['o_end']): ov += 1
print('aspect span token-lengths:', dict(sorted(alen.items())[:12]))
print('opinion span token-lengths:', dict(sorted(olen.items())[:12]))
print(f'aspect/opinion OVERLAPPING spans in same quad: {ov}/{tot_eo} ({ov/max(tot_eo,1):.1%})')
# category ↔ sentiment degeneracy (is sentiment nearly constant per category?)
cs = collections.defaultdict(collections.Counter)
for v in splits.values():
    for r in v:
        for q in r['quads']: cs[q['category']][q['sentiment']] += 1
skew = [(c, max(d.values())/sum(d.values()), sum(d.values())) for c, d in cs.items()]
skew.sort(key=lambda x: -x[2])
print('\ncategory → dominant-sentiment share (top 10 by freq):')
for c, s, n in skew[:10]: print(f'  {c}: {s:.0%} of {n}')